# 1.1 面向 Ascend 的 SpMV 后端开发与优化原型

> 实验主入口：[Ascend C FP32 SpMV 与混合精度 Host 原型实验手册](EXPERIMENT_GUIDE.md)。FP32 baseline 由真实 Ascend C RTC Kernel 执行；FP16/BF16/persistent 路径仍明确标为 Host Prototype。

第4章调用现成 ACL 算子；本章转向一个可替换后端的开发实验。原 `Ascend-SpMV` 工程以 CSR SpMV 为对象，依次实现 FP32 baseline、nnz-aware partition、FP16/BF16 mixed precision 和 persistent context，并用统一 benchmark 比较 CPU single、OpenMP16 与这些后端。

## 实验条件

当前工程的 FP32 baseline 已使用 ACL RTC 编译并启动真实 Ascend C Kernel；CSR、x、y 均由 `aclrtMalloc` 分配并通过 H2D/D2H 传输。旧 FP16/BF16/persistent 循环已统一标为 Host Prototype，其历史 CSV 不作为当前 NPU 结果。

## 前置要求

- 完成第2章 CSR/OpenMP 与第4章 ACL 调用
- 理解 backend、warmup/repeat 和相对误差

## 学习目标

- 追踪 `ISpmvBackend::prepare/run` 与 benchmark 调用链
- 比较 FP32、FP16、BF16 和 persistent 数据路径
- 解释 nnz-aware partition、cold start、warm total 和 footprint
- 从 README 历史 CSV 中区分实验记录与当前源码执行边界

## 环境检查

直接检查本节需要的运行环境；若检查失败，请先在对应 CPU/NPU 节点加载课程要求的工具链。


In [ ]:
import platform, shutil
print("Python:", platform.python_version())
print("CMake:", shutil.which("cmake"))
print("正式路径：Ascend C FP32 RTC；FP16/BF16/persistent 仍为 Host Prototype")


## 章节内容

- [01.01_chapter_intro](01.01_chapter_intro.ipynb)：实验定位与边界
- [01.02_backend_interface_and_execution_model](01.02_backend_interface_and_execution_model.ipynb)：后端接口和执行模型
- [01.03_precision_and_partition](01.03_precision_and_partition.ipynb)：mixed precision 与 nnz 分区
- [01.04_persistent_context](01.04_persistent_context.ipynb)：persistent CSR 生命周期
- [01.05_build_validation_and_performance](01.05_build_validation_and_performance.ipynb)：构建、正确性与历史结果
- [01.06_chapter_test](01.06_chapter_test.ipynb)：后端审计综合实践

## 观察点

课程副本保留原 README、CMake、benchmark、后端源码和两份历史 CSV。下一节从实际接口出发，不预设它是标准 Ascend C 工程。

## 实验工程说明与本章任务

本章实验工程位于 `src/ascend_spmv/`。`include/spmv.hpp` 定义 backend；`npu_spmv*.cpp` 实现 FP32/FP16/BF16 原型；`npu_spmv_context.cpp` 实现 nnz-aware partition 和 persistent；benchmark/脚本负责运行；CSV 是历史记录。

### 本章实验任务

构建 benchmark → 追踪接口 → 比较精度表示与误差 → 检查 nnz-aware 边界 → 区分 cold/warm → 分析历史 CSV → 说明执行边界。

所有路径均相对 Notebook 当前目录。先检查环境，再运行真实工程；历史结果只用于观察趋势。

### 完成标准

能够指出核心源码和脚本职责，完成可用环境内的构建/诊断，并按“Backend、Precision、Cold、Warm、Total、Compression、Balance、Error”记录证据。
